# Modelo BiGRU-TCN: Entrenamiento y Evaluación

In [1]:
import os
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import optuna

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.eval.losses import QLIKELoss
from src.eval.scorer import VolatilityScorer
from src.models.bigru_tcn_wrapper import BiGRU_TCNModel
from src.trainer.walk_forward import WalkForwardTrainer
from src.utils import trainer_toolkit as tt
from src.utils import ts_toolkit as tst
from src.utils import test_toolkit as ts

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "bigru_tcn"
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any()
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any()

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)


:::{admonition} Resumen
:class: info
- La configuración óptima del modelo BiGRU-TCN empleó una ventana de contexto de 7 días, 32 unidades ocultas en el bloque BiGRU, 32 canales y 3 niveles en la TCN, 2 capas densas de 256 unidades, *dropout* de 0.176, tasa de aprendizaje de 0.0004 y *weight decay* de $2.0 \times 10^{-5}$.
- En entrenamiento, el modelo siguió la trayectoria general de la volatilidad realizada con leve sobreestimación en periodos de calma relativa y subestimación en picos agudos como el de la COVID-19, presentando residuos de colas pesadas mejor descritos por una distribución $t$ de Student que por la normal.
- En evaluación fuera de muestra (*walk-forward*, 72.47 minutos para 953 observaciones), el QLIKE aumentó por encima del umbral de tolerancia de 0.05 frente a entrenamiento ($\Delta_{\text{abs}} = +0.1838$), acompañado de saltos en HMSE y HMAE y persistencia de efectos ARCH en el rezago 10, sin autocorrelación ni dependencia no lineal en dimensión 2
- En el episodio de volatilidad de 2026, el modelo siguió la dinámica general, pero subestimó la magnitud del pico principal de marzo y mostró ligeras desviaciones hacia arriba en los cuantiles superiores de la cola derecha, evidenciando la necesidad de profundizar en el estudio de las arquitecturas recurrentes y optimizar los tiempos de inferencia.
:::

## Introducción

El modelo híbrido *BiGRU-TCN* (`BiGRU_TCNModel`) combina una capa recurrente bidireccional (*Bidirectional Gated Recurrent Unit*, BiGRU) con bloques convolucionales temporales causales dilatados (*Temporal Convolutional Network*, `TCNBlock`) para capturar dependencias secuenciales y patrones multi-escala a lo largo de una ventana histórica de longitud $L$ (`look_back`). Mientras la rama BiGRU sintetiza representaciones contextuales en ambos sentidos temporales sobre la ventana de entrada, los bloques TCN aplican convoluciones causales con dilatación exponencial para extraer estructuras locales y de mayor alcance sin recurrencia convolucional adicional.

En series financieras, esta combinación permite modelar relaciones no lineales entre retornos históricos estandarizados y la dinámica de la varianza del crudo Brent, integrando la codificación secuencial de la BiGRU con la extracción jerárquica de características de la TCN bajo un esquema unificado.

El objetivo de esta sección es presentar, entrenar y evaluar el modelo `BiGRU_TCNModel` para pronosticar la varianza realizada de Rogers-Satchell ($RS$) de los futuros del crudo Brent (`BZ=F`) bajo el protocolo del *benchmark*. Se analiza el efecto de la longitud de ventana (`look_back`), las unidades ocultas de la BiGRU (`bigru_hidden`), los canales y niveles de dilatación de la TCN (`tcn_channels`, `tcn_levels`, `kernel_size`, `num_convs`), la estructura de la cabeza densa (`dense_units`, `num_dense_layers`) y la regularización (`dropout`, `learning_rate` y `weight_decay`) sobre el desempeño predictivo, manteniendo las mismas condiciones de información, pérdida **QLIKE** y evaluación *walk-forward* que el resto de modelos del proyecto.

### Arquitectura

La arquitectura de `BiGRU_TCNNetwork` se organiza en tres etapas secuenciales: una capa `nn.GRU` bidireccional, una pila de bloques `TCNBlock` con dilatación creciente y una cabeza densa multicapa (`head`) con activación *Softplus* y piso numérico positivo.

Dado un tensor de entrada $\mathbf{X}_t \in \mathbb{R}^{L \times 1}$ (donde $L$ es `look_back` sobre la serie univariada de retornos estandarizados), la capa BiGRU (`num_layers=1`, `bidirectional=True`, `batch_first=True`) produce una secuencia de estados ocultos concatenados de dimensión $2 \cdot h_{\text{gru}}$ (donde $h_{\text{gru}}$ es `bigru_hidden`):

$$
\mathbf{H}_{\text{BiGRU}} = \text{BiGRU}(\mathbf{X}_t) \in \mathbb{R}^{L \times 2h_{\text{gru}}}.
$$

Para adecuar la representación secuencial al formato canal-por-tiempo esperado por la red convolucional unidimensional, se reordena la salida de la BiGRU a $\widetilde{\mathbf{H}}_{\text{BiGRU}} \in \mathbb{R}^{2h_{\text{gru}} \times L}$. Esta representación atraviesa una secuencia de $K$ bloques convolucionales (`tcn_levels`), donde cada bloque $k \in \{0, \dots, K-1\}$ emplea dilatación $d_k = 2^k$, tamaño de núcleo `kernel_size`, `num_convs` convoluciones internas, regularización `dropout` y proyección hacia `tcn_channels` (tras el primer bloque, todos los bloques conservan `tcn_channels` canales):

$$
\mathbf{Z}^{(0)} = \widetilde{\mathbf{H}}_{\text{BiGRU}}, \qquad \mathbf{Z}^{(k)} = \text{TCNBlock}_{k}\left(\mathbf{Z}^{(k-1)};\, d_k = 2^k\right), \quad k = 1, \dots, K.
$$

De la salida final de la TCN, $\mathbf{Z}^{(K)} \in \mathbb{R}^{C_{\text{tcn}} \times L}$ (con $C_{\text{tcn}} = \text{tcn\_channels}$), se extrae el estado correspondiente al último paso temporal $\mathbf{z}_L^{(K)} \in \mathbb{R}^{C_{\text{tcn}}}$ y se propaga por una cabeza densa de `num_dense_layers` capas ocultas (`Linear` + `ReLU` + `Dropout` cuando `dropout > 0.0`), donde la primera capa emplea `dense_units` ($u_1$) y las capas subsecuentes utilizan $u_j = \max(16, \lfloor u_1 / 2 \rfloor)$ unidades, seguida de una proyección lineal final a un escalar $y_t \in \mathbb{R}$:

$$
y_t = \text{Head}\left(\mathbf{z}_L^{(K)}\right).
$$

Para garantizar que la varianza pronosticada sea estrictamente positiva y numéricamente estable, $y_t$ se transforma mediante la función *Softplus* sumada a un piso $\varepsilon = 10^{-8}$:

$$
\widehat{RS}_t = \text{Softplus}(y_t) + \varepsilon = \ln\left(1 + e^{y_t}\right) + \varepsilon,
$$

donde $\boldsymbol{\theta}$ reúne todos los parámetros entrenables de la BiGRU, los bloques TCN y la cabeza densa.

Siguiendo el protocolo del *benchmark*, los retornos logarítmicos diarios $r_t$ se estandarizan mediante transformación Z-score ajustada exclusivamente sobre el segmento de entrenamiento de cada *fold* para evitar fuga de información, mientras que la varianza realizada de Rogers-Satchell se escala mediante el factor de objetivo (`target_scale`). El modelo opera bajo un esquema de predicción de un paso adelante (*one-step-ahead forecasting*) y hereda de `BasePyTorchVolatilityModel` la integración con el *walk-forward* (`retrain_freq=1`), optimización con Adam, *weight decay*, *Dropout* y parada temprana, minimizando la pérdida asimétrica **QLIKE**:

$$
\boldsymbol{\theta}^{*} = \arg\min_{\boldsymbol{\theta}} \frac{1}{N} \sum_{i=1}^{N} \mathcal{L}_{\text{QLIKE}}\left(RS_{i}, \widehat{RS}_{i}\right),
$$

donde $N$ es el número de ventanas de entrenamiento y $RS_i$ denota la varianza de Rogers-Satchell objetivo. La implementación se desarrolla en `PyTorch` con diferenciación automática.

## Entrenamiento

Para la selección de hiperparámetros del modelo `BiGRU_TCNModel`, se mantuvo el mismo protocolo experimental (*Walk-Forward*) y el mismo presupuesto computacional que en los modelos paramétricos, recurrentes y convolucionales. A diferencia del GARCH y el EGARCH, la búsqueda de Optuna exploró la arquitectura híbrida compuesta por la capa recurrente bidireccional (`BiGRU`), los bloques convolucionales dilatados (`TCNBlock`), la cabeza densa (`head`) y sus parámetros de regularización.

Para adaptar la serie temporal al formato requerido por la red, se construyó una matriz de rezagos (*lag matrix*) para cada ventana del esquema *Walk-Forward*. Siguiendo los principios de validación cruzada para series temporales (`timeseriescv`), cada fila de esta matriz contenía una secuencia histórica de longitud `look_back`, la cual se utilizó como vector de entrada para predecir el paso inmediatamente posterior. Este enfoque garantizó la integridad temporal de los datos y evitó cualquier tipo de fuga de información futura (*look-ahead bias*) durante el entrenamiento y el reentrenamiento (`retrain_freq=1`, `seed=42`).

El espacio de búsqueda incluyó la longitud de la ventana de entrada (`look_back`), las unidades ocultas de la BiGRU (`bigru_hidden`), los canales y niveles de dilatación de la TCN (`tcn_channels`, `tcn_levels`, `kernel_size`, `num_convs`), la configuración de la cabeza densa (`dense_units`, `num_dense_layers`), la tasa de abandono (`dropout`), la tasa de aprendizaje (`learning_rate`) y la penalización $\ell_2$ (`weight_decay`). Para asegurar una cobertura adecuada de la ventana histórica sin incrementar innecesariamente la profundidad, el número de niveles convolucionales (`tcn_levels`) se definió dentro de un rango acotado a partir de `look_back`, `kernel_size` y `num_convs`, con un límite superior de $\min(\text{min\_levels} + 2, 5)$.

El entrenamiento se configuró con un máximo de 200 épocas y una paciencia de 20 para la parada temprana interna de la red, bajo un esquema de búsqueda con `n_trials=50` y `early_stop_rounds=15`. El desempeño se siguió evaluando mediante la pérdida QLIKE sobre los conjuntos de validación.

In [ ]:
import math
import optuna


def bigru_tcn_search_space(trial: optuna.Trial) -> dict:
    look_back = trial.suggest_categorical("look_back", [7, 14, 21, 28])
    kernel_size = trial.suggest_categorical("kernel_size", [2, 3])
    num_convs = trial.suggest_categorical("num_convs", [1, 2, 3])

    min_levels = math.ceil(
        math.log2(1 + (look_back - 1) / (num_convs * (kernel_size - 1)))
    )
    max_levels = min(min_levels + 2, 5)

    return {
        "look_back": look_back,
        "bigru_hidden": trial.suggest_categorical("bigru_hidden", [32, 64, 128]),
        "tcn_channels": trial.suggest_categorical("tcn_channels", [32, 64, 128]),
        "tcn_levels": trial.suggest_int("tcn_levels", min_levels, max_levels),
        "kernel_size": kernel_size,
        "num_convs": num_convs,
        "dense_units": trial.suggest_categorical("dense_units", [32, 64, 128, 256]),
        "num_dense_layers": trial.suggest_categorical("num_dense_layers", [1, 2]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.3),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        "max_epochs": 200,
        "patience": 20,
        "retrain_freq": 1,
        "seed": 42,
    }


exec_id = 1
current_warmup = 1200 + (exec_id * 10)

exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=BiGRU_TCNModel,
    scorer=VolatilityScorer(),
    output_dir=exec_dir,
    n_splits=5,
    warmup=current_warmup,
    embargo=1,
    n_trials=50,
    suggest_params_fn=bigru_tcn_search_space,
    early_stop_rounds=15,
)

trainer.run(y_train, rs_train)

Aunque el protocolo experimental contemplaba un presupuesto de 50 ensayos (`n_trials=50`), el estudio de Optuna registró 14 ensayos en esta ejecución, completando 12 ensayos válidos (85.7 %), con 1 ensayo podado y 0 fallidos, y alcanzando un mejor valor de QLIKE de 0.4127. La presencia de un ensayo podado refleja el descarte temprano de una configuración menos favorable durante la búsqueda, mientras que la arquitectura híbrida *BiGRU-TCN* con bloques convolucionales dilatados y parada temprana permitió completar la mayoría de las evaluaciones realizadas.

La búsqueda en Optuna tomó aproximadamente 19 horas y 8 minutos, mientras que el ajuste y entrenamiento del modelo bajo el esquema *Walk-Forward* totalizó aproximadamente 4.30 horas, con un promedio de unos 4.16 minutos por partición. Este tiempo de ajuste evidencia el costo computacional de combinar una capa recurrente bidireccional y bloques convolucionales dilatados entrenados con Adam y retropropagación frente a modelos más ligeros, conservando la viabilidad operativa dentro del protocolo *Walk-Forward*.

In [2]:
resumen_data: list[dict] = []

exec_id = 1
model_tag = "BiGRU_TCN"
study_name_override = None

exec_dir = Path(
    r"C:\Users\jcami\Documents\brent-crude-nlp-market-dynamics\results\bigru_tcn"
) / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_qlike: float | str = "-"

csv_files = sorted(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    if "train_time_sec" in df_csv.columns:
        tiempo_total_rep = float(df_csv["train_time_sec"].sum())
        tiempo_promedio_fold = float(df_csv["train_time_sec"].mean())
    num_folds = len(df_csv)
else:
    print(f"No se encontró el CSV en {exec_dir}")

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        available_studies = optuna.get_all_study_names(storage=storage_url)

        if study_name_override and study_name_override in available_studies:
            target_study = study_name_override
        elif available_studies:
            target_study = available_studies[-1]
        else:
            target_study = f"{model_tag}_study"

        study = optuna.load_study(study_name=target_study, storage=storage_url)

        total_trials = len(study.trials)
        completos = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
        podados = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)
        fallidos = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.FAIL)

        if completos > 0 and study.best_value is not None:
            mejor_qlike = round(float(study.best_value), 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")
else:
    print(f"No se encontró la BD en {db_path}")

resumen_data.append(
    {
        "Exec": exec_id,
        "Total Trials": total_trials,
        "Completos": completos,
        "Podados": podados,
        "Fallidos": fallidos,
        "Mejor QLIKE": mejor_qlike,
        "Tiempo Total (s)": round(tiempo_total_rep, 2),
        "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3),
    }
)

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Exec,Total Trials,Completos,Podados,Fallidos,Mejor QLIKE,Tiempo Total (s),Tiempo por Fold (s)
1,14,12,1,0,0.4127,15466.9000,249.4660


El ranking de las cinco mejores configuraciones muestra variación en el horizonte de ventana (`look_back` entre 7 y 28 días) y en la profundidad convolucional (`tcn_levels` entre 3 y 5), mientras que `bigru_hidden` se sitúa en 32 o 64, `kernel_size` en 2 o 3, `num_convs` en 1 a 3, `dense_units` entre 32 y 256, `num_dense_layers` en 1 o 2 y `learning_rate` entre 0.0004 y 0.0034. Esta distribución refleja cómo la combinación de una capa recurrente bidireccional con convoluciones causales dilatadas opera bajo distintos horizontes históricos y configuraciones de capacidad.

In [6]:
csv_path = exec_dir / "BiGRU_TCNModelFT_results.csv"
if not csv_path.exists():
    fallback_csvs = sorted(exec_dir.glob("BiGRU_TCNModelFT*_results.csv")) or sorted(
        exec_dir.glob("*_results.csv")
    )
    if not fallback_csvs:
        raise FileNotFoundError(f"No se encontró archivo de resultados en {exec_dir}")
    csv_path = fallback_csvs[0]

df_bigru = pd.read_csv(csv_path)

if "learning_rate" in df_bigru.columns:
    df_bigru["lr"] = df_bigru["learning_rate"].apply(lambda x: f"{x:.4f}")
if "weight_decay" in df_bigru.columns:
    df_bigru["wd"] = df_bigru["weight_decay"].apply(lambda x: f"{x:.1e}")
if "dropout" in df_bigru.columns:
    df_bigru["drop_fmt"] = df_bigru["dropout"].apply(lambda x: f"{x:.3f}")

candidatas_agrupar = [
    "look_back",
    "bigru_hidden",
    "tcn_channels",
    "tcn_levels",
    "kernel_size",
    "num_convs",
    "dense_units",
    "num_dense_layers",
    "drop_fmt",
    "lr",
    "wd",
]
cols_agrupar = [c for c in candidatas_agrupar if c in df_bigru.columns]

metric_col = "qlike" if "qlike" in df_bigru.columns else "val_loss"

df_agg = (
    df_bigru.groupby(cols_agrupar, as_index=False)[metric_col]
    .agg(mean="mean", std="std")
)
df_agg["std"] = df_agg["std"].fillna(0.0)
df_agg["QLIKE"] = df_agg.apply(
    lambda row: f"{row['mean']:.3f} ± {row['std']:.3f}", axis=1
)

renombres = {
    "look_back": "Lookback",
    "bigru_hidden": "BiGRU Hidden",
    "tcn_channels": "TCN Ch",
    "tcn_levels": "TCN Levels",
    "kernel_size": "Kernel",
    "num_convs": "Convs",
    "dense_units": "Dense",
    "num_dense_layers": "Dense Layers",
    "drop_fmt": "Dropout",
    "lr": "LR",
    "wd": "Weight Decay",
}

df_top5 = (
    df_agg.sort_values(by="mean", ascending=True)
    .head(5)
    .rename(columns={k: v for k, v in renombres.items() if k in df_agg.columns})
)

cols_finales = [
    v for v in renombres.values() if v in df_top5.columns
] + ["QLIKE"]

df_final = df_top5[cols_finales]
tt.display_styled_df(df=df_final, align_left_first=False, decimals=3)

Lookback,BiGRU Hidden,TCN Ch,TCN Levels,Kernel,Convs,Dense,Dense Layers,Dropout,LR,Weight Decay,QLIKE
7,32,32,3,2,1,256,2,0.176,0.0004,2.0e-05,0.413 ± 0.068
28,64,16,5,2,2,256,2,0.117,0.0004,1.3e-05,0.419 ± 0.049
14,32,32,3,3,3,32,1,0.159,0.0013,2.7e-06,0.433 ± 0.088
21,64,16,3,2,3,128,1,0.110,0.0034,4.5e-05,0.443 ± 0.073
21,32,32,5,2,1,256,2,0.222,0.0016,1.5e-05,0.458 ± 0.175


Entre las dos primeras configuraciones, la diferencia en QLIKE es reducida frente a la dispersión observada entre particiones, por lo que ambas muestran un desempeño cercano en validación. La primera configuración presenta el menor QLIKE medio con una estructura más compacta en su rama recurrente y convolucional frente a la segunda, que reduce ligeramente la variabilidad entre particiones a cambio de mayor ventana y profundidad convolucional.

In [ ]:
best_bigru_tcn_model = BiGRU_TCNModel(
    look_back=7,
    bigru_hidden=32,
    tcn_channels=32,
    tcn_levels=3,
    kernel_size=2,
    num_convs=1,
    dense_units=256,
    num_dense_layers=2,
    dropout=0.176,
    learning_rate=0.0004,
    weight_decay=2.0e-5,
    max_epochs=200,
    patience=20,
    retrain_freq=1,
    seed=42,
)

start_time = time.time()
best_bigru_tcn_model.fit(y_train, rs_train)
end_time = time.time()

elapsed_time = end_time - start_time

El entrenamiento del modelo sobre toda la muestra tuvo una duración de 4.67 segundos. En la gráfica de aprendizaje, tras un pico pronunciado en entrenamiento cerca de la época 4 y oscilaciones iniciales en ambas trayectorias, el QLIKE descendió con rapidez y se estabilizó alrededor de la época 37, donde se seleccionaron los mejores pesos y el proceso continuó hasta cerca de la época 57.

A diferencia de lo observado en arquitecturas con mayor ajuste relativo sobre la muestra de entrenamiento, la curva de validación se mantuvo por debajo de la de entrenamiento durante la convergencia y en la meseta posterior. Este patrón sugiere una buena capacidad de generalización sin sobreajuste severo, mientras que la estabilización posterior indica que el modelo capturó el nivel base de la varianza condicional con margen acotado para refinar picos extremos.

In [11]:
import pandas as pd

history_csv_path = exec_dir / "bigru_tcn_training_history.csv"
history_df = pd.read_csv(history_csv_path)

if "epoch" in history_df.columns:
    history_df = history_df.drop(columns=["epoch"])

history_from_csv = {
    col: history_df[col].dropna().tolist()
    for col in history_df.columns
}

n_epochs = len(history_df)
x_max = max(n_epochs + 1, 2)
x_step = max(1, round(x_max / 11))

tt.plot_learning_curve(
    history=history_from_csv,
    title="Curva de Aprendizaje",
    ylab="Quasi-Likelihood (QLIKE)",
    xlab_pad=14, ylab_pad=14,
    line_width=2.0, title_size=14,
    label_size=14, tick_size=12,
    legend_size=14, ylim=(0, 8.5),
    xlim=(0, x_max), y_step=0.5,
    x_step=x_step, legend_bbox=(0.95, 0.95)
)

Para evaluar el ajuste dentro de la muestra de entrenamiento, se realizó un pase hacia adelante (*forward pass*) sobre la serie completa de entrenamiento: primero se transformó `y_train` con el `StandardScaler` ajustado durante `fit` y se escaló el objetivo de varianza, luego se construyó la matriz de rezagos con `TimeSeriesDataset` usando ventanas de 7 días (`look_back = 7`) y finalmente se evaluó en modo `eval()` sin gradientes ni regularización. Las varianzas estimadas se reescalaron a su unidad original y se alinearon con los datos reales descartando las observaciones iniciales correspondientes a la ventana.

In [ ]:
best_bigru_tcn_model.net.eval()
look_back = best_bigru_tcn_model.context_length
target_scale = getattr(best_bigru_tcn_model, "target_scale", 1.0)

y_train_scaled = best_bigru_tcn_model.scaler.transform(
    np.asarray(y_train).reshape(-1, 1)
).flatten()
rs_train_scaled = np.asarray(rs_train) * target_scale

train_eval_dataset = TimeSeriesDataset(
    y_train_scaled,
    rs_train_scaled,
    look_back,
)
train_eval_loader = DataLoader(
    train_eval_dataset,
    batch_size=getattr(best_bigru_tcn_model, "batch_size", 128),
    shuffle=False,
)

preds_train_list = []
with torch.no_grad():
    for x_batch, _ in train_eval_loader:
        x_batch = x_batch.to(best_bigru_tcn_model.device)
        preds_batch = best_bigru_tcn_model.net(x_batch).squeeze(-1).detach().cpu().numpy()
        preds_train_list.append(preds_batch)

preds_train_var = np.concatenate(preds_train_list) / target_scale
n_preds = len(preds_train_var)

start_idx = len(y_train) - n_preds
dates_aligned = df_train.index[start_idx : start_idx + n_preds]
y_train_aligned = np.asarray(y_train)[start_idx : start_idx + n_preds]
rs_train_aligned = np.asarray(rs_train)[start_idx : start_idx + n_preds]

df_train_preds = pd.DataFrame(
    {
        "y_true": y_train_aligned,
        "rs_true": rs_train_aligned,
        "y_pred": preds_train_var,
    },
    index=dates_aligned,
)

train_output_path = results_dir / "bigru_tcn_train_predictions.csv"
df_train_preds.to_csv(train_output_path, index=True)

A partir de las varianzas condicionales predichas $\left(\hat{\sigma}_t^2\right)$ y de los retornos observados $\left(y_t\right)$, se evaluó el ajuste del modelo sobre el conjunto de entrenamiento mediante los residuos estandarizados:

$$
z_t = \frac{y_t}{\sqrt{\hat{\sigma}_t^2}}
$$

El diagnóstico sobre los residuos estandarizados mostró ausencia de autocorrelación lineal significativa tras el ajuste de Bonferroni en las pruebas de Ljung-Box para los rezagos 10, 20 y 30 ($p_{\text{adj}} \ge .338$), así como ausencia de dependencia no lineal en la prueba BDS en dimensión 2 ($p = .766$). A diferencia de otros modelos, las pruebas ARCH-LM no rechazaron la hipótesis nula de homocedasticidad condicional en los rezagos 10, 20 y 30 ($p_{\text{adj}} = 1.000$), lo que indica ausencia de agrupamiento residual de volatilidad dentro de la muestra de entrenamiento. Asimismo, las pruebas de Jarque-Bera y Lilliefors rechazaron la normalidad de los residuos ($p < .001$), reflejando colas pesadas características de los retornos del crudo Brent.

In [3]:
import numpy as np
import pandas as pd

train_output_path = exec_dir / "bigru_tcn_train_predictions.csv"

df_train_preds = pd.read_csv(train_output_path)
rs_train_aligned = df_train_preds["rs_true"].to_numpy()
preds_train_var = df_train_preds["y_pred"].to_numpy()

n_preds = len(preds_train_var)
y_arr = np.asarray(y_train).reshape(-1)
y_train_aligned = y_arr[-n_preds:]

std_residuals_train = y_train_aligned / np.sqrt(np.maximum(preds_train_var, 1e-12))
std_residuals_train = std_residuals_train[np.isfinite(std_residuals_train)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_train,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística",
        "Estadístico",
        "p-valor",
        "p-adj",
        "Conclusión (α = 0.05)",
    ),
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--"
)


def _format_p(x):
    if pd.isna(x):
        return "--"
    if x < 0.001:
        return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted


df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),6.0602,.810,1.000,Sin Autocorrelación
Ljung-Box (Lag 20),27.8626,.113,.338,Sin Autocorrelación
Ljung-Box (Lag 30),37.6795,.158,.474,Sin Autocorrelación
ARCH-LM (Lag 10),9.9532,.445,1.000,Sin Efectos ARCH
ARCH-LM (Lag 20),20.8541,.406,1.000,Sin Efectos ARCH
ARCH-LM (Lag 30),29.4255,.495,1.000,Sin Efectos ARCH
Jarque-Bera,1309.0035,< .001,--,Residuos No Normales
Lilliefors,0.0604,< .001,--,Residuos No Normales
BDS (Dim 2),0.2972,.766,--,Sin Estructura No Lineal


Los residuos estandarizados del modelo mostraron un ajuste general a una distribución $t$ de Student ($\nu = 4.4525$). En los cuantiles extremos de ambas colas se observaron desviaciones respecto a la referencia teórica, con los cuantiles superiores empíricos situándose por debajo de los teóricos. Estas discrepancias en las colas implicaron que la aproximación $t$ de Student sobrestimó ligeramente la magnitud de los eventos extremos en comparación con la cola empírica de los residuos, reflejando un ajuste más preciso en la región central que en los valores atípicos más severos. Las pruebas de bondad de ajuste (Kolmogorov-Smirnov: $p = .5319$; Cramér-von Mises: $p = .3458$) no rechazaron la hipótesis nula de ajuste a la distribución $t$.

In [14]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_train),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.8),
    legend_loc="upper left"
)

La prueba de razón de verosimilitud (LRT) rechazó la hipótesis nula de igualdad de ajuste frente a la distribución normal ($p < .001$). Asimismo, los criterios de información mostraron una reducción de $\Delta\text{AIC} = 217.61$ y $\Delta\text{BIC} = 211.93$ a favor de la distribución $t$ de Student. Estos resultados apoyaron que la distribución $t$ capturara con mayor precisión las colas pesadas de los residuos estandarizados del modelo en la muestra de entrenamiento en comparación con el ajuste Gaussiano.

In [15]:
_= tst.distribution_fit_table(
    pd.Series(std_residuals_train),
    column_titles=(
        "Distribución", 
        "Log-Likelihood", 
        "AIC", 
        "BIC", 
        "Estadístico LRT", 
        "p-valor LRT"
    )
)

Distribución,Log-Likelihood,AIC,BIC,Estadístico LRT,p-valor LRT
Normal,"-2,891.09","5,786.19","5,797.55",-,-
t-Student,"-2,781.29","5,568.58","5,585.62",219.61,< .001


La inspección de las predicciones dentro de la muestra de entrenamiento indica que la volatilidad estimada por el modelo BiGRU-TCN ($\hat{\sigma}_t$) sigue la trayectoria general de la volatilidad realizada ($\sigma_R$) y responde al incremento de volatilidad durante la crisis de la COVID-19 (enero a junio de 2020). En los periodos de calma relativa, como gran parte de 2014 a 2018, $\hat{\sigma}_t$ se ubica por encima de $\sigma_R$ (sobrestimación leve y suavizada del nivel base). Por el contrario, en los picos extremos de volatilidad, como los repuntes de marzo y abril de 2020 en el recuadro ampliado, $\hat{\sigma}_t$ subestima la magnitud máxima alcanzada por $\sigma_R$.

In [17]:
n_preds = len(preds_train_var)
start_idx = len(y_train) - n_preds

dates_aligned = df_train.index[start_idx : start_idx + n_preds]
rs_volatilidad_aligned = np.sqrt(np.maximum(np.asarray(rs_train)[start_idx : start_idx + n_preds], 0.0))
pred_volatilidad = np.sqrt(np.maximum(preds_train_var, 0.0))

mask_crisis = (dates_aligned >= "2020-01-01") & (dates_aligned <= "2020-06-30")

ts.plot_volatility_fit(
    dates=dates_aligned,
    rs_variance=rs_volatilidad_aligned,
    pred_variance=pred_volatilidad,
    title="Ajuste dentro de la Muestra de Entrenamiento (BiGRU-TCN)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad BiGRU-TCN $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.7),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)

## Evaluación

La evaluación del rendimiento de un modelo de volatilidad requiere contrastar su ajuste sobre los datos observados con su capacidad predictiva en observaciones no vistas. Mientras que el conjunto de entrenamiento permite verificar la convergencia del algoritmo sobre la dinámica histórica, la evaluación fuera de muestra determina el desempeño del modelo ante nuevos regímenes de mercado.

Para diagnosticar posibles problemas de generalización, se aplicaron los criterios pre-registrados del protocolo experimental empleado en los modelos de aprendizaje profundo evaluados hasta el momento, a partir de la diferencia entre las métricas de prueba y entrenamiento:

$$
\Delta_{\text{abs}} = M_{\text{test}} - M_{\text{train}}
$$

Se fijó un umbral de tolerancia de $|\Delta_{\text{abs}}| = 0.05$ para las métricas evaluadas, de modo que discrepancias por encima de este margen señalan cambios relevantes entre el ajuste en muestra y el desempeño fuera de muestra.

En el modelado de volatilidad con pérdida QLIKE, esta lectura debe matizarse por la asimetría de la función de pérdida, que castiga con mayor severidad la subestimación de la varianza que su sobreestimación. Un modelo con sesgo defensivo o de mayor varianza predicha puede registrar un QLIKE menor en prueba que en entrenamiento sin que ello implique una mejor capacidad estructural, sino una coincidencia favorable entre el nivel predicho y el régimen de volatilidad del periodo de prueba. Por ello, el análisis de $\Delta_{\text{abs}}$ se complementa con la inspección de las trayectorias predichas y el diagnóstico de residuos estandarizados.

Las predicciones fuera de muestra del modelo BiGRU-TCN se generaron mediante el método `forecast`, aplicando el esquema secuencial de evaluación con ventana expansiva y parámetro `retrain_freq = 1` sin fuga de información, y devolviendo las varianzas condicionales en la escala original para su comparación directa con la volatilidad realizada.

In [ ]:
start_time = time.time()
preds_test = best_bigru_tcn_model.forecast(y_seq=y_test, rs_seq=rs_test)
preds_time = time.time() - start_time

print(f"Tiempo de pronóstico: {preds_time / 60:.2f} minutos")

df_preds = pd.DataFrame({
    "date": df_test.index,
    "rs_true": rs_test,
    "y_pred": preds_test,
})

output_path = exec_dir / "bigru_tcn_test_predictions.csv"
df_preds.to_csv(output_path, index=False)

La generación de pronósticos fuera de muestra mediante el esquema de ventana expansiva y reentrenamiento periódico tuvo una duración total de 72.47 minutos para las 953 observaciones del conjunto de prueba, lo que equivale a un tiempo promedio de 0.08 minutos (unos 4.56 segundos) por observación. Este tiempo es sustancialmente superior al de los modelos paramétricos, cuyo cómputo toma milisegundos, reflejando el costo computacional de actualizar y reajustar los pesos de la red neuronal a lo largo de la ventana de evaluación.

El diagnóstico de los residuos estandarizados fuera de muestra mostró ausencia de autocorrelación lineal significativa tras el ajuste de Bonferroni en las pruebas de Ljung-Box para los rezagos 10, 20 y 30 ($p_{\text{adj}} \ge .132$), así como ausencia de dependencia no lineal en la prueba BDS en dimensión 2 ($p = .403$). En las pruebas ARCH-LM, el rezago 10 presentó efectos ARCH significativos tras el ajuste de Bonferroni ($p_{\text{adj}} = .037$), mientras que en los rezagos 20 y 30 no se rechazó la hipótesis nula de homocedasticidad condicional ($p_{\text{adj}} \ge .413$). Asimismo, las pruebas de Jarque-Bera y Lilliefors rechazaron la normalidad de los residuos ($p < .001$), reflejando colas pesadas en los retornos del crudo Brent fuera de muestra.

In [4]:
preds_path = exec_dir / "bigru_tcn_test_predictions.csv"
df_preds_test = pd.read_csv(preds_path)
preds_test_var = df_preds_test["y_pred"].values

std_residuals_oos = y_test / np.sqrt(np.maximum(preds_test_var, 1e-12))
std_residuals_oos = std_residuals_oos[np.isfinite(std_residuals_oos)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_oos,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística",
        "Estadístico",
        "p-valor",
        "p-adj",
        "Conclusión (α = 0.05)",
    ),
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--"
)


def _format_p(x):
    if pd.isna(x):
        return "--"
    if x < 0.001:
        return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted


df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),12.3815,.260,.781,Sin Autocorrelación
Ljung-Box (Lag 20),31.9215,.044,.132,Sin Autocorrelación
Ljung-Box (Lag 30),40.8695,.089,.267,Sin Autocorrelación
ARCH-LM (Lag 10),22.5936,.012,.037,Efectos ARCH Significativos
ARCH-LM (Lag 20),26.9176,.138,.413,Sin Efectos ARCH
ARCH-LM (Lag 30),30.0464,.463,1.000,Sin Efectos ARCH
Jarque-Bera,642.1140,< .001,--,Residuos No Normales
Lilliefors,0.0505,< .001,--,Residuos No Normales
BDS (Dim 2),-0.8369,.403,--,Sin Estructura No Lineal


La persistencia de un efecto ARCH remanente en el rezago 10 indica que la varianza condicional estimada no absorbe por completo la agrupación de volatilidad de corto plazo en la muestra de prueba. Dado que el objetivo del modelo es capturar la dinámica temporal de la volatilidad y estandarizar adecuadamente los choques, esta heterocedasticidad residual señala una reacción parcial o rezagada ante cambios bruscos de régimen antes de que la varianza se disipe en rezagos mayores.

La inspección visual de los residuos estandarizados fuera de muestra muestra un ajuste general adecuado a la distribución $t$ de Student con $\nu = 4.4433$ grados de libertad. Los cuantiles muestrales siguen de cerca la diagonal teórica en el centro y en gran parte de las colas, aunque se observan leves desviaciones hacia arriba en algunos cuantiles superiores (alrededor de 3 a 5). En coherencia con esta correspondencia general, las pruebas formales de bondad de ajuste (Kolmogorov-Smirnov: $p = .8232$; Cramér-von Mises: $p = .8389$) no rechazaron la hipótesis nula de pertenencia a la distribución $t$.

In [19]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_oos),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (t-Student)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.7),
    legend_loc="upper left"
)

La prueba de razón de verosimilitud (LRT) rechazó la hipótesis nula de igualdad de ajuste frente a la distribución normal en los residuos estandarizados en el conjunto de prueba ($p < .001$). Los criterios de información mostraron una reducción de $\Delta\text{AIC} = 111.33$ y $\Delta\text{BIC} = 106.47$ a favor de la distribución $t$ de Student. Estos resultados apoyaron que la distribución $t$ capturara con mayor precisión las colas pesadas de los residuos estandarizados fuera de muestra en comparación con el ajuste Gaussiano.

In [20]:
_= tst.distribution_fit_table(
    pd.Series(std_residuals_oos),
    column_titles=(
        "Distribución", 
        "Log-Likelihood", 
        "AIC", 
        "BIC", 
        "Estadístico LRT", 
        "p-valor LRT"
    )
)

Distribución,Log-Likelihood,AIC,BIC,Estadístico LRT,p-valor LRT
Normal,"-1,530.91","3,065.83","3,075.55",-,-
t-Student,"-1,474.25","2,954.50","2,969.08",113.32,< .001


El modelo BiGRU-TCN excedió el umbral de tolerancia de $|\Delta_{\text{abs}}| = 0.05$ en QLIKE con un incremento pronunciado entre entrenamiento y prueba ($\Delta_{\text{abs}} = +0.1838$). Este deterioro fuera de muestra también se observa en el modelo GRU ($\Delta_{\text{abs}} = +0.1012$), un resultado relevante porque evidencia que las arquitecturas recurrentes presentan dificultades para generalizar con precisión la dinámica de volatilidad del crudo Brent ante nuevos regímenes.

En el resto de las métricas del BiGRU-TCN, los errores de escala directa (CRPS, MSE, RMSE y MAE) y las pérdidas *pinball* mantuvieron diferencias reducidas y cercanas a cero, mientras que las métricas ajustadas por heterocedasticidad registraron incrementos en prueba en HMSE ($\Delta_{\text{abs}} = +9.4016$) y HMAE ($\Delta_{\text{abs}} = +0.3824$), superando el umbral de tolerancia y siguiendo el mismo patrón de aumento del GRU en HMSE ($\Delta_{\text{abs}} = +4.9655$) y HMAE ($\Delta_{\text{abs}} = +0.1248$) bajo la asimetría de la pérdida frente a subestimaciones de varianza. Este comportamiento sugiere un sobreajuste a la dinámica histórica de entrenamiento o una adaptación insuficiente a la varianza de prueba, en contraste con arquitecturas puramente convolucionales como la TCN.

In [21]:
look_back = 7
rs_train_aligned = rs_train[look_back:]

ts.display_train_test_comparison(
    rs_train=rs_train_aligned,
    preds_train=preds_train_var,
    rs_test=rs_test,
    preds_test=preds_test_var,
    column_titles=(
        "Métrica",
        "Entrenamiento",
        "Prueba",
        "Diferencia",
    ),
)

Métrica,Entrenamiento,Prueba,Diferencia
QLIKE,0.4565,0.6402,+0.1838
CRPS,0.0127,0.0126,-0.0001
Pinball (τ=0.05),0.0031,0.0026,-0.0004
Pinball (τ=0.95),0.0014,0.0020,+0.0006
MSE,0.0000,0.0000,+0.0000
RMSE,0.0018,0.0021,+0.0004
MAE,0.0005,0.0005,-0.0000
HMSE,0.7454,10.1471,+9.4016
HMAE,0.6196,1.0021,+0.3824


La inspección visual de las predicciones fuera de muestra respalda los resultados numéricos. La subestimación de los picos agudos de volatilidad realizada ($\sigma_R$), como el repunte de marzo de 2026 en el recuadro ampliado, coincide con el incremento de QLIKE ($\Delta_{\text{abs}} = +0.1838$) y de HMSE ($\Delta_{\text{abs}} = +9.4016$) debido a la penalización asimétrica que recibe la subestimación de la varianza. Asimismo, la respuesta suavizada y con cierto rezago de $\hat{\sigma}_t$ frente a cambios abruptos en el nivel de volatilidad es coherente con la persistencia de efectos ARCH en el rezago 10 detectada en los residuos estandarizados.

In [5]:
df_preds_test["date"] = pd.to_datetime(df_preds_test["date"])

rs_test_vol = np.sqrt(np.maximum(df_preds_test["rs_true"], 0.0))
pred_test_vol = np.sqrt(np.maximum(df_preds_test["y_pred"], 0.0))

mask_crisis = (df_preds_test["date"] >= "2026-02-01") & (df_preds_test["date"] <= "2026-06-30")

ts.plot_volatility_fit(
    dates=df_preds_test["date"],
    rs_variance=rs_test_vol,
    pred_variance=pred_test_vol,
    title="Ajuste Fuera de la Muestra (BiGRU-TCN)",
    ylab="Volatilidad",
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$",
    label_pred=r"Volatilidad BiGRU-TCN $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35),
    legend_bbox=(0.01, 0.98),
    zoom_slice=mask_crisis,
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Pico de Volatilidad (2026)",
    zoom_bbox=(0.25, 0.35, 0.40, 0.40),
    title_size=14,
    label_size=14,
    tick_size=12,
    legend_size=14,
)

## Conclusiones

La implementación y evaluación del modelo BiGRU-TCN permitieron contrastar el desempeño de esta arquitectura híbrida recurrente y convolucional para el pronóstico de la volatilidad diaria del crudo Brent. En el conjunto de prueba, la pérdida QLIKE aumentó por encima del umbral de tolerancia de $|\Delta_{\text{abs}}| = 0.05$ respecto a la muestra de entrenamiento ($\Delta_{\text{abs}} = +0.1838$), un comportamiento que coincide con el deterioro fuera de muestra observado en el modelo GRU ($\Delta_{\text{abs}} = +0.1012$) y refuerza la necesidad de investigar con mayor detalle por qué las arquitecturas recurrentes enfrentan dificultades para generalizar con precisión la dinámica de volatilidad del crudo Brent ante nuevos regímenes.

En el diagnóstico fuera de muestra, los residuos estandarizados no presentaron autocorrelación lineal significativa tras el ajuste de Bonferroni en las pruebas de Ljung-Box para los rezagos 10, 20 y 30 ni estructura no lineal en la prueba BDS en dimensión 2. Sin embargo, las pruebas ARCH-LM detectaron efectos ARCH significativos en el rezago 10 ($p_{\text{adj}} = .037$), aunque no en los rezagos 20 y 30, indicando una absorción incompleta de la heterocedasticidad condicional de corto plazo. Asimismo, las pruebas de Jarque-Bera y Lilliefors rechazaron la normalidad, y los residuos mantuvieron colas pesadas mejor descritas por la distribución $t$ de Student ($\nu = 4.4433$) que por la normal, con pruebas de bondad de ajuste (Kolmogorov-Smirnov y Cramér-von Mises) favorables a la distribución $t$.

La asimetría de la función de pérdida QLIKE, que penaliza con mayor severidad la subestimación de la varianza realizada frente a la sobreestimación, amplificó el efecto de las subestimaciones en picos agudos como el repunte de marzo de 2026 y se reflejó en el incremento fuera de muestra de las métricas ajustadas por heterocedasticidad (HMSE y HMAE). El esquema secuencial de pronóstico con reentrenamiento periódico (`retrain_freq = 1`) tomó 72.47 minutos para las 953 observaciones de prueba (0.08 minutos o 4.56 segundos por observación en promedio), un costo computacional muy superior al de los modelos paramétricos. En conjunto, los resultados muestran que el BiGRU-TCN captura la trayectoria general de la volatilidad y modeliza razonablemente la distribución de colas pesadas, pero requiere mayor precisión en la calibración de picos extremos, menor heterocedasticidad residual de corto plazo y una investigación más profunda sobre el comportamiento de las arquitecturas recurrentes y la eficiencia del tiempo de inferencia antes de consolidarse frente a alternativas más ligeras o puramente convolucionales.